In [12]:
# imports 

import pprint
from vame.pipeline import VAMEPipeline
import vame
from pathlib import Path

In [13]:
# Set File Location
source_software = "DeepLabCut"

videos = ["Subject 2 - female GO Vehicle OFT_DLC.mp4"]
poses_estimations = ["Subject 2 - female GO Vehicle OFT_DLC.h5"]
fps = 30

print(videos)
print(poses_estimations)
print(fps)

['Subject 2 - female GO Vehicle OFT_DLC.mp4']
['Subject 2 - female GO Vehicle OFT_DLC.h5']
30


In [14]:
config_file, config_data = vame.init_new_project(
    project_name="Subject 2 - Female Go Vehicle OFT - Test",
    poses_estimations=poses_estimations,
    source_software="DeepLabCut",
    fps=fps,
)

2026-02-18 09:29:21.113 INFO  --- [MainThread] vame.initialize_project.new : 109 : Created "C:\Users\zuria\Kaplan\VAME\Subject 2 - Female Go Vehicle OFT - Test\data"
2026-02-18 09:29:21.115 INFO  --- [MainThread] vame.initialize_project.new : 109 : Created "C:\Users\zuria\Kaplan\VAME\Subject 2 - Female Go Vehicle OFT - Test\data\raw"
2026-02-18 09:29:21.115 INFO  --- [MainThread] vame.initialize_project.new : 109 : Created "C:\Users\zuria\Kaplan\VAME\Subject 2 - Female Go Vehicle OFT - Test\data\processed"
2026-02-18 09:29:21.117 INFO  --- [MainThread] vame.initialize_project.new : 109 : Created "C:\Users\zuria\Kaplan\VAME\Subject 2 - Female Go Vehicle OFT - Test\results"
2026-02-18 09:29:21.118 INFO  --- [MainThread] vame.initialize_project.new : 109 : Created "C:\Users\zuria\Kaplan\VAME\Subject 2 - Female Go Vehicle OFT - Test\model"
2026-02-18 09:29:21.119 INFO  --- [MainThread] vame.initialize_project.new : 109 : Created "C:\Users\zuria\Kaplan\VAME\Subject 2 - Female Go Vehicle OFT

In [15]:
print(config_data)
ds_path = Path(config_data["project_path"]) / "data" / "raw" / f"{config_data['session_names'][0]}.nc"
vame.io.load_poses.load_vame_dataset(ds_path)

{'vame_version': '0.12.0', 'project_name': 'Subject 2 - Female Go Vehicle OFT - Test', 'project_path': 'C:\\Users\\zuria\\Kaplan\\VAME\\Subject 2 - Female Go Vehicle OFT - Test', 'creation_datetime': '2026-02-18T17:29:21+00:00', 'session_names': ['Subject 2 - female GO Vehicle OFT_DLC'], 'project_random_state': 42, 'all_data': 'yes', 'keypoints': ['nose', 'left_ear', 'right_ear', 'left_ear_tip', 'right_ear_tip', 'left_eye', 'right_eye', 'neck', 'mid_back', 'mouse_center', 'mid_backend', 'mid_backend2', 'mid_backend3', 'tail_base', 'tail1', 'tail2', 'tail3', 'tail4', 'tail5', 'left_shoulder', 'left_midside', 'left_hip', 'right_shoulder', 'right_midside', 'right_hip', 'tail_end', 'head_midpoint'], 'egocentric_data': False, 'pose_confidence': 0.99, 'robust': True, 'iqr_factor': 4, 'savgol_filter': True, 'savgol_length': 5, 'savgol_order': 2, 'test_fraction': 0.1, 'model_name': 'VAME', 'pretrained_model': 'None', 'pretrained_weights': False, 'num_features': 54, 'batch_size': 256, 'max_epoc

<xarray.Dataset> Size: 12MB
Dimensions:      (time: 18330, space: 2, keypoints: 27, individuals: 1)
Coordinates:
  * time         (time) float64 147kB 0.0 0.03333 0.06667 ... 610.9 610.9 611.0
  * space        (space) <U1 8B 'x' 'y'
  * keypoints    (keypoints) <U14 2kB 'nose' 'left_ear' ... 'head_midpoint'
  * individuals  (individuals) <U7 28B 'animal0'
Data variables:
    position     (time, space, keypoints, individuals) float64 8MB -1.0 ... 4...
    confidence   (time, keypoints, individuals) float64 4MB -1.0 -1.0 ... 0.8697
Attributes:
    source_software:  DeepLabCut
    ds_type:          poses
    fps:              30.0
    time_unit:        seconds
    source_file:      Subject 2 - female GO Vehicle OFT_DLC.h5

In [16]:
vame.preprocessing(
    config=config_data,
    centered_reference_keypoint="nose",
    orientation_reference_keypoint="tail_base",
    run_lowconf_cleaning=True,   
    run_outlier_cleaning=True,
    # savgol filtering constantly errors due to nans or infs
    run_savgol_filtering=False,
    run_egocentric_alignment=True
)

2026-02-18 09:29:21.570 INFO  --- [MainThread] vame.preprocessing.preprocessing : 67 : Cleaning low confidence data points...
2026-02-18 09:29:21.571 INFO  --- [MainThread] vame.preprocessing.cleaning : 46 : Cleaning low confidence data points. Confidence threshold: 0.99
2026-02-18 09:29:21.572 INFO  --- [MainThread] vame.preprocessing.cleaning : 49 : Session: Subject 2 - female GO Vehicle OFT_DLC
2026-02-18 09:29:21.715 INFO  --- [MainThread] vame.preprocessing.preprocessing : 78 : Egocentrically aligning and centering...
2026-02-18 09:29:21.717 INFO  --- [MainThread] vame.preprocessing.alignment : 78 : Egocentric alignment with references: nose and tail_base
2026-02-18 09:29:21.717 INFO  --- [MainThread] vame.preprocessing.alignment : 85 : Session: Subject 2 - female GO Vehicle OFT_DLC
2026-02-18 09:29:22.124 INFO  --- [MainThread] vame.preprocessing.preprocessing : 91 : Cleaning outliers using IQR method...
2026-02-18 09:29:22.126 INFO  --- [MainThread] vame.preprocessing.cleaning :

'position_processed'

In [17]:
vame.create_trainset(
    config=config_data,
    keypoints_to_exclude=[
        "nose", "tail_base", # Alignment points (Must exclude!)
    ]
)
vame.train_model(config=config_data)


2026-02-18 09:31:18.712 INFO  --- [MainThread] vame.model.create_training : 320 : Creating training dataset...
2026-02-18 09:31:18.914 INFO  --- [MainThread] vame.model.create_training : 187 : Session Subject 2 - female GO Vehicle OFT_DLC: test chunk 15795:17628 (length 1833)
2026-02-18 09:31:18.923 INFO  --- [MainThread] vame.model.create_training : 240 : Metadata file saved for feature provenance tracking
2026-02-18 09:31:18.924 INFO  --- [MainThread] vame.model.create_training : 241 : Metadata: C:\Users\zuria\Kaplan\VAME\Subject 2 - Female Go Vehicle OFT - Test\data\train\metadata.json
2026-02-18 09:31:18.925 INFO  --- [MainThread] vame.model.create_training : 243 : Length of train data: 16497
2026-02-18 09:31:18.925 INFO  --- [MainThread] vame.model.create_training : 244 : Length of test data: 1833
2026-02-18 09:31:18.926 INFO  --- [MainThread] vame.model.create_training : 245 : Number of features: 50
2026-02-18 09:31:18.974 INFO  --- [MainThread] vame.model.create_training : 335 :

_LinAlgError: linalg.svd: The algorithm failed to converge because the input matrix is ill-conditioned or has too many repeated singular values (error code: 256).

In [ ]:
vame.evaluate_model(config=config_data)

In [ ]:
vame.segment_session(
    config=config_data,
    overwrite_segmentation=False,
    overwrite_embeddings=False,
)

In [ ]:
vame.community(
    config=config_data,
    cut_tree=2,
)

In [ ]:
from vame.visualization import visualize_preprocessing_cloud

visualize_preprocessing_cloud(config=config_data)
from vame.visualization import plot_loss

plot_loss(config=config_data)
from IPython.display import Image, display

eval_plot_path = Path(config_data["project_path"]) / "model" / "evaluate" / "future_reconstruction.png"
display(Image(filename=eval_plot_path))
from vame.visualization import visualize_motif_thresholding

visualize_motif_thresholding(
    config=config_data,
    segmentation_algorithm="hmm",
    threshold=1.0,
)
from vame.visualization import visualize_hierarchical_tree

visualize_hierarchical_tree(
    config=config_data,
    segmentation_algorithm="hmm",
)


In [ ]:
vame.motif_videos(config=config_data)
vame.community_videos(config_data)

In [ ]:
from vame.visualization import visualize_umap

visualize_umap(
    config=config_data,
    show_figure="plotly",
)
from vame.visualization import visualize_motif_thresholding

visualize_motif_thresholding(
    config=config_data,
    segmentation_algorithm="hmm",
    threshold=1.0,
)